# Kind-LM Mechanistic Probe

The BabyLM 2025 paper (*"Once Upon a Time"*, Mayer Martins et al.) found that interactive PPO-based
learning provides massive sample-efficiency gains, but only after a **20M–50M word pretraining
threshold**. Nobody knows *why*. This notebook uses mechanistic interpretability to find out.

### Analysis list:
1. **Induction Head Emergence** - Does the threshold coincide with the birth of induction heads?
2. **PPO Causal Isolation** - Does interaction training selectively strengthen threshold-born circuits?
3. **Entity Coreference Tracking** - Can narrative coherence be localised to specific heads?
4. **Logit Lens KL Depth** - Does the model "decide" earlier in the network post-threshold?

## Environment Setup


In [ ]:
import subprocess, sys

packages = [
    "numpy<2.0.0",      # pinned to prevent C API binary incompatibility (Expected 96, got 88)
    "transformer_lens",
    "einops",
    "plotly",
    "jaxtyping",
    "transformers",
    "pandas",
    "kaleido",          # for plotly static image export
    "scipy",            # for FDR correction
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("All packages installed.")

## Imports & Global Config

In [ ]:
import os
import gc
import json
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import false_discovery_control
from transformers import GPT2LMHeadModel, AutoTokenizer, GPT2Config
from transformer_lens import HookedTransformer

#  Directories 
os.makedirs("results/figures",  exist_ok=True)
os.makedirs("results/data",     exist_ok=True)

#  Device 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥  Running on: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

#  Checkpoint registry (all verified against live HuggingFace, 2025-04) 
REPO_BASE = "llm-slice/blm-gpt2s-90M-s42"

# Dense coverage around the hypothesised threshold (20M–50M words)
PRETRAINING_REVISIONS = [
    "chck_1M",
    "chck_5M",
    "chck_10M",
    "chck_20M",
    "chck_30M",
    "chck_40M",
    "chck_50M",
    "chck_60M",
    "chck_100M",
    "chck_200M",
    "chck_1000M",
]

# PPO interaction checkpoints - separate repos, canonical seed=42
PPO_REPOS = {
    "50M+PPO":  "llm-slice/blm-gpt2s-90M-s42_chck_50M_ppo-1000K-seed42",
    "200M+PPO": "llm-slice/blm-gpt2s-90M-s42_chck_200M_ppo-1000K-seed42",
}

# Expected custom vocabulary size - assert on every load to catch mismatches
EXPECTED_VOCAB = 16000

# Story prompt used across analyses (domain-matched to BabyLM training data)
STORY_PROMPT = "Once upon a time in a faraway land, there lived a"

print("Config loaded.")

## Robust Model Loading Utility

**Critical:** `HookedTransformer.from_pretrained("gpt2")` would load embedding
matrices sized for vocab=50,257. Because the BabyLM models use vocab=16,000,
we must load the native HuggingFace model first, then pass it to
`HookedTransformer` via `hf_model=` so TransformerLens inherits the correct
weight shapes. An `assert` guards every call.

In [ ]:
def load_model(repo_id: str, revision: str = "main", random_init: bool = False) -> HookedTransformer:
    """
    Load a BabyLM checkpoint into a TransformerLens HookedTransformer.

    Parameters
    -
    repo_id      : HuggingFace repo string
    revision     : branch name (e.g. "chck_50M") or "main"
    random_init  : if True, creates a randomly-initialised model with the same
                   architecture (used as the chck_0M architectural baseline)

    Returns
    -
    HookedTransformer on DEVICE with vocab=16000 verified
    """
    print(f"  ↳ Loading {'RANDOM INIT' if random_init else revision} from {repo_id}")

    # Use chck_1M to source the config for random init (it has tokenizer files)
    config_revision = revision if not random_init else "chck_1M"

    tokenizer = AutoTokenizer.from_pretrained(repo_id, revision=config_revision)

    if random_init:
        config   = GPT2Config.from_pretrained(repo_id, revision=config_revision)
        hf_model = GPT2LMHeadModel(config)           # random weights, correct architecture
    else:
        hf_model = GPT2LMHeadModel.from_pretrained(repo_id, revision=revision)

    tl_model = HookedTransformer.from_pretrained(
        "gpt2",
        hf_model=hf_model,
        tokenizer=tokenizer,
        device=DEVICE,
        n_ctx=hf_model.config.n_positions,
        d_vocab=hf_model.config.vocab_size
    )

    assert tl_model.cfg.d_vocab == EXPECTED_VOCAB, (
        f"❌ Vocab mismatch! Got {tl_model.cfg.d_vocab}, expected {EXPECTED_VOCAB}. "
        f"Check revision '{revision}' - tokenizer must come from a chck_* branch."
    )

    tl_model.eval()
    return tl_model


def release(model: HookedTransformer):
    """Aggressively free GPU memory after each model is done."""
    model.reset_hooks(clear_contexts=True)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def vram_gb() -> float:
    if DEVICE == "cuda":
        return torch.cuda.memory_allocated() / 1e9
    return 0.0




## Analysis 1 - High-Resolution Induction Head Mapping

**Hypothesis:** The pretraining threshold coincides with the emergence of
induction heads - specialised circuits that implement in-context copying
essential for narrative coherence (`[A][B]...[A] → [B]`).

**Falsifying outcome:** If induction scores grow with no
inflection point between 20M–60M words the threshold is not tied to
induction head formation.

In [ ]:
def get_induction_scores(model: HookedTransformer) -> torch.Tensor:
    """
    Compute per-head induction score using a repeated random token sequence.

    Returns
    -
    Tensor of shape [n_layers, n_heads] with scores in [0, 1].
    """
    model.reset_hooks()

    # A sequence of the form [A B C D E | A B C D E]
    # chosen to be safely within the 16K custom vocabulary
    prefix = [101, 250, 310, 420, 533]
    seq    = prefix + prefix
    tokens = torch.tensor([seq], device=DEVICE)

    _, cache = model.run_with_cache(
        tokens,
        names_filter=lambda name: "pattern" in name,    # only cache attention patterns
    )

    n_layers = model.cfg.n_layers
    n_heads  = model.cfg.n_heads
    scores   = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        pattern = cache["pattern", layer][0]            # [n_heads, q_pos, k_pos]
        for head in range(n_heads):
            # Attention from position 5 (second A) back to position 1 (first B)
            scores[layer, head] = pattern[head, 5, 1].item()

    model.reset_hooks()
    return scores                                       # stays on CPU for storage


#  Run across all checkpoints 
induction_rows = []

# Architectural baseline (random init, 0 words of training)
print("  [0/12] Random init baseline …")
m = load_model(REPO_BASE, random_init=True)
scores = get_induction_scores(m)
induction_rows.append({"checkpoint": "0M_random", "max_score": scores.max().item(),
                       "mean_score": scores.mean().item(), "scores_json": scores.tolist()})
release(m)

# Pretraining checkpoints
for i, rev in enumerate(PRETRAINING_REVISIONS, 1):
    print(f"  [{i}/{len(PRETRAINING_REVISIONS)}] {rev} …")
    m = load_model(REPO_BASE, revision=rev)
    scores = get_induction_scores(m)
    induction_rows.append({"checkpoint": rev.replace("chck_", ""),
                            "max_score": scores.max().item(),
                            "mean_score": scores.mean().item(),
                            "scores_json": scores.tolist()})
    release(m)

df_induction = pd.DataFrame(induction_rows)
df_induction.to_csv("results/data/analysis_1_induction_scores.csv", index=False)

#  Visualise 
fig1 = px.line(
    df_induction,
    x="checkpoint",
    y="max_score",
    markers=True,
    title="Analysis 1: Peak Induction Score Across Pretraining Checkpoints",
    labels={"checkpoint": "Pretraining Corpus Size (words)", "max_score": "Max Induction Score"},
    template="plotly_white",
)
fig1.add_vrect(x0="20M", x1="50M", fillcolor="orange", opacity=0.15,
               annotation_text="Hypothesised threshold", annotation_position="top left")
fig1.write_html("results/figures/analysis_1_induction_trajectory.html")
fig1.show()

## Analysis 2 - Causal Isolation of the PPO Effect

**Hypothesis:** PPO fine-tuning selectively amplifies the induction heads that
emerged at the threshold rather than creating fundamentally new circuits.

We take matched pairs: same pretraining base, ±PPO interaction, and compute
the Δ-induction map.

**Falsifying outcome:** If Δ-induction ≈ 0 across all heads at the 50M level,
PPO is not driving circuit-level changes in induction behaviour.

In [ ]:
print("▶ Analysis 2: PPO Causal Isolation (50M vs 50M+PPO)")

ppo_records = []

for label, (base_rev, ppo_repo) in {
    "50M":  ("chck_50M",  PPO_REPOS["50M+PPO"]),
    "200M": ("chck_200M", PPO_REPOS["200M+PPO"]),
}.items():
    m_base = load_model(REPO_BASE, revision=base_rev)
    scores_base = get_induction_scores(m_base)
    release(m_base)

    m_ppo  = load_model(ppo_repo, revision="main")
    scores_ppo = get_induction_scores(m_ppo)
    release(m_ppo)

    delta = scores_ppo - scores_base          # [n_layers, n_heads]
    torch.save(delta, f"results/data/analysis_2_delta_{label}vsPPO.pt")

    # Flatten to layer/head/delta rows for visualisation
    n_layers, n_heads = delta.shape
    for layer in range(n_layers):
        for head in range(n_heads):
            ppo_records.append({
                "base": label,
                "layer": layer,
                "head": head,
                "delta_induction": delta[layer, head].item(),
            })

df_ppo = pd.DataFrame(ppo_records)
df_ppo.to_csv("results/data/analysis_2_ppo_delta.csv", index=False)

#  Visualise 
for base_label in ["50M", "200M"]:
    sub = df_ppo[df_ppo["base"] == base_label]
    pivot = sub.pivot(index="layer", columns="head", values="delta_induction")
    fig2 = px.imshow(
        pivot,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        title=f"Analysis 2: Δ Induction Score (PPO − Base) @ {base_label} pretraining",
        labels={"color": "Δ Induction Score"},
        template="plotly_white",
    )
    fig2.write_html(f"results/figures/analysis_2_delta_{base_label}.html")
    fig2.show()

## Analysis 3 - Entity Coreference Tracking

**Hypothesis:** Storytelling RL is effective only once the model can reliably
track character entities across a narrative.

We use a simple coreference probe rather than the standard IOI circuit
(which assumes standard GPT-2 vocabulary and single-token names).

**Validation guard:** We explicitly check that probe words are single tokens
in the custom vocabulary before running; if not, the result is logged as
`"multi-token - skipped"` rather than silently failing.

In [ ]:
def safe_single_token(model: HookedTransformer, word: str):
    """Return token id if word is a single token, else raise ValueError."""
    tokens = model.to_tokens(word, prepend_bos=False)[0]
    if len(tokens) > 1:
        readable = model.to_str_tokens(tokens)
        raise ValueError(f"'{word}' is {len(tokens)} tokens: {readable}")
    return tokens[0].item()


COREFERENCE_PROBES = [
    {
        "prompt":  "The boy gave the ball to the girl . The ball belonged to the",
        "correct": " boy",
        "foil":    " girl",
    },
    {
        "prompt":  "The dog followed the cat into the wood . The cat ran away and hid from the",
        "correct": " dog",
        "foil":    " cat",
    },
    {
        "prompt":  "The king gave the ring to the queen . The ring belonged to the",
        "correct": " king",
        "foil":    " queen",
    },
    {
        "prompt":  "The man and the woman walked down the street . The woman smiled and waved at the",
        "correct": " man",
        "foil":    " woman",
    },
]

print("▶ Analysis 3: Entity Coreference Logit Difference")
coref_rows = []

for rev in ["chck_10M", "chck_50M", "chck_1000M"]:
    print(f"  {rev} …")
    m  = load_model(REPO_BASE, revision=rev)
    for probe in COREFERENCE_PROBES:
        try:
            correct_id = safe_single_token(m, probe["correct"])
            foil_id    = safe_single_token(m, probe["foil"])
            tokens     = m.to_tokens(probe["prompt"])
            logits, _  = m.run_with_cache(tokens, names_filter=lambda n: False)  # no cache needed
            logit_diff = (logits[0, -1, correct_id] - logits[0, -1, foil_id]).item()
        except ValueError as e:
            logit_diff = None
            print(f"    ⚠ Skipped probe (tokenisation): {e}")

        coref_rows.append({
            "checkpoint":   rev.replace("chck_", ""),
            "probe_correct": probe["correct"].strip(),
            "logit_diff":   logit_diff,
        })
    release(m)

df_coref = pd.DataFrame(coref_rows).dropna(subset=["logit_diff"])
df_coref.to_csv("results/data/analysis_3_coreference_logits.csv", index=False)

#  Visualise 
fig3 = px.bar(
    df_coref,
    x="checkpoint",
    y="logit_diff",
    color="probe_correct",
    barmode="group",
    title="Analysis 3: Entity Coreference Logit Difference Across Checkpoints",
    labels={"checkpoint": "Pretraining Size (words)", "logit_diff": "Logit Diff (correct − foil)"},
    template="plotly_white",
)
fig3.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="Chance level")
fig3.write_html("results/figures/analysis_3_coreference.html")
fig3.show()

## Analysis 4 - Logit Lens KL-Divergence Decision Depth

**Hypothesis:** Well-trained models commit to their next-token prediction
earlier in the network (lower layer) than undertrained ones - reflecting a
more efficient representational pipeline.

Metric: KL divergence between each layer's unembed prediction and the final
model output. Depth = the layer where KL first drops below ε = 0.5.

**Implementation note:** We use `model.ln_final` to apply the correct
LayerNorm before multiplying by `W_U`, avoiding the systematic DLA bias that
arises when LayerNorm is skipped.

In [ ]:
EPS_THRESHOLD = 0.5   # KL threshold for "model has committed"

def logit_lens_kl(model: HookedTransformer, prompt: str):
    """
    For every layer, compute KL( layer_l_prediction || final_prediction ).
    Returns list of KL values, one per layer.
    """
    tokens = model.to_tokens(prompt)

    _, cache = model.run_with_cache(
        tokens,
        names_filter=lambda n: "resid_post" in n,
    )
    final_logits  = model(tokens)[0, -1]
    final_probs   = F.softmax(final_logits, dim=-1)

    kl_per_layer = []
    for layer in range(model.cfg.n_layers):
        resid   = cache["resid_post", layer][0, -1]          # [d_model]
        ln_out  = model.ln_final(resid.unsqueeze(0)).squeeze(0)
        logits_l = ln_out @ model.W_U
        probs_l  = F.softmax(logits_l, dim=-1)
        kl       = F.kl_div(probs_l.log(), final_probs, reduction="sum").item()
        kl_per_layer.append(kl)

    return kl_per_layer


print("▶ Analysis 4: Logit Lens KL Depth")
kl_rows = []

for rev in ["chck_10M", "chck_50M", "chck_1000M"]:
    print(f"  {rev} …")
    m      = load_model(REPO_BASE, revision=rev)
    klds   = logit_lens_kl(m, STORY_PROMPT)
    commit = next((i for i, v in enumerate(klds) if v < EPS_THRESHOLD), m.cfg.n_layers)
    release(m)

    for layer, kl in enumerate(klds):
        kl_rows.append({
            "checkpoint":   rev.replace("chck_", ""),
            "layer":        layer,
            "kl_divergence": kl,
            "commit_layer": commit,
        })

df_kl = pd.DataFrame(kl_rows)
df_kl.to_csv("results/data/analysis_4_logit_lens_kl.csv", index=False)

#  Visualise 
fig4 = px.line(
    df_kl,
    x="layer",
    y="kl_divergence",
    color="checkpoint",
    markers=True,
    title="Analysis 4: Logit Lens KL-Divergence per Layer",
    labels={"layer": "Layer Index", "kl_divergence": "KL Divergence (vs final output)"},
    template="plotly_white",
)
fig4.add_hline(y=EPS_THRESHOLD, line_dash="dot", line_color="red",
               annotation_text=f"Commit threshold (ε={EPS_THRESHOLD})")
fig4.write_html("results/figures/analysis_4_logit_lens.html")
fig4.show()

## Summary Table

In [ ]:
commit_lookup = (
    df_kl.groupby("checkpoint")["commit_layer"].first().to_dict()
)
peak_ind = (
    df_induction.set_index("checkpoint")["max_score"].to_dict()
)

summary_rows = []
for rev in ["10M", "50M", "1000M"]:
    peak_score = peak_ind.get(rev, peak_ind.get(f"{rev}", None))
    summary_rows.append({
        "Checkpoint":         f"{rev} words",
        "Peak Induction Score": f"{peak_ind.get(rev, '-'):.3f}" if isinstance(peak_ind.get(rev), float) else "-",
        "KL Commit Layer":    commit_lookup.get(rev, "-"),
    })

# PPO delta
for base_label in ["50M", "200M"]:
    sub = df_ppo[df_ppo["base"] == base_label]
    top_delta = sub["delta_induction"].abs().max()
    summary_rows.append({
        "Checkpoint":         f"{base_label}+PPO (delta)",
        "Peak Induction Score": f"Δ max = {top_delta:.3f}",
        "KL Commit Layer":    "n/a",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("results/data/summary_table.csv", index=False)
print(df_summary.to_string(index=False))
print("Summary table saved.")

## Package & Download All Results


In [ ]:
import shutil
shutil.make_archive("Kind_LM_Project_B_Results", "zip", "results")
try:
    from google.colab import files
    files.download("Kind_LM_Project_B_Results.zip")
except ImportError:
    pass
